In [1]:
#Connecticut River at Thompsonville, CT 01184000 1928 - current data

library(lubridate)
library(ggplot2)
#library(dplyr)
library(plyr)
library(tibble)
library(readr)
library(gridExtra)
library(tidyverse)
library(dataRetrieval) #USGS data retrieval #MED downloaded package on 2025-07-30 v2.7.20 in r-usgs environment
library(cowplot) #plot_grid
library(grid)

# For MED inputs
input_dir <- "~//OneDrive/git-repos/cQ_analysis/winter-discharge/Megan2025_WinLamoille/Megan2025_WinLamoille/"
output_dir <- "~//OneDrive/git-repos/LCBP-interannual-EMMAs/Output/Hydrographs/"

#file.choose()
# Carol original below:
#qdat<-read_csv("/Users/carol/Proposals/2019 DOE SBR/2019 SBR/Full proposal/VT discharge/Winooski04290500_discharge_R.csv")
# MED current below: 
#qdat <- read.csv(file.path(input_dir, "Winooski04290500_discharge_R.csv"))

#Connecticut River discharge at Thompsonville, CT
#siteNumber <- "04290500"
siteNumber <- "01184000"
WInfo <- readNWISsite(siteNumber)
parameterCd <- "00060" #Discharge

# Raw daily data:
qdat <- as_tibble(readNWISdv(
  siteNumber, parameterCd,
  "1929-10-01", "2025-01-03"
))

#qdatDate,format = "%m/%d/%Y")
#need year only
qdat$Year<-as.integer(format(as.Date(qdat$Date, format="%d/%m/%Y"),"%Y"))
#need julian day
qdat$DOY<-as.integer(strftime(qdat$Date, format = "%j"))

#cfs to cms
qdatX_00060_00003*0.028316847

qdat.29.60<-subset(qdat, Date > "1928-12-31" & Date < "1961-01-01")
#cut off after august
qdat.29.60.a<-subset(qdat.29.60, DOY<213)

#mean
qavg.29.60<-ddply(qdat.29.60.a, c("DOY"), function(df)
  return(c(q.min=min(df$Qcms), q.max = max(dfQcms,na.rm=TRUE), q.se=sd(df$Q,na.rm=TRUE)/sqrt(nrow(df)))))
ggplot(qavg.29.60, aes(x=DOY, y=q.avg)) + geom_line() + theme_bw() +
  labs(y="Streamflow (cms)", x="Day of Year")+ 
  geom_ribbon(aes(ymin=q.min, ymax=q.max, x=DOY, fill = "band"), alpha = 0.3)

qdat.29.60.a$month.day<-format(qdat.29.60.a$Date, format="%m-%d")
qdat.29.60.a$month<-month(qdat.29.60.a$Date)
qdat.29.60.a$mday<-mday(qdat.29.60.a$Date)
labels(qdat.29.60.a)
head(qdat.29.60.a)
qavg.29.60<-ddply(qdat.29.60.a, c("month.day"), function(df)
  return(c(q.min=min(df$Qcms), q.max = max(dfQcms,na.rm=TRUE), q.se=sd(df$Q,na.rm=TRUE)/sqrt(nrow(df)))))
nrow(qavg.29.60)
labels(qavg.29.60)
head(qavg.29.60)
class(qavg.29.60$month.day)
qavg.29.60$month.day<-as.Date(qavg.29.60$month.day,format="%m-%d")

dat3<-data.frame(x1 = as.Date(c("2020-01-01")),
                 x2 = as.Date(c("2020-03-20")),
                 y1 = -Inf,
                 y2 = Inf)
head(qavg.29.60)
norm.wint<-ggplot(qavg.29.60, aes(x=month.day, y=q.avg)) + geom_line() + theme_bw() +
  labs(y="Streamflow (cms)", x="") +
  scale_x_date(date_breaks = "1 month", date_minor_breaks = "1 week", date_labels = "%b") #+
  # geom_rect(data = dat3,
  #           aes(xmin = x1, xmax = x2, ymin = y1, ymax = y2),
  #           fill = "blue", alpha = 0.2, 
  #           inherit.aes = FALSE) #+
  #annotate("text",x=as.Date("2019-12-28"), y=180, label= "(a)") 
  #geom_ribbon(aes(ymin=q.min, ymax=q.max, x=DOY, fill = "band"), alpha = 0.3)
norm.wint

head(qdat)
unique(qdat$Year)
temp1<-subset(qdat, Year>2017 & Year<2019)
temp1<-subset(temp1, DOY<213)
nrow(temp1)

#dat1<-data.frame(x1 = decimal_date(as.Date(c("2018-01-01"))),
#           x2 = decimal_date(as.Date(c("2018-04-01"))),
#           y1 = -Inf,
#           y2 = Inf)
dat2<-data.frame(x1 = as.Date(c("2018-01-01")),
                 x2 = as.Date(c("2018-03-20")),
                 y1 = -Inf,
                 y2 = Inf)

one.wint<-ggplot(temp1, aes(x=Date, y=Qcms)) + geom_line() + theme_bw() +
  labs(y="Streamflow (cms)", x="") +
  scale_x_date(date_breaks = "1 month", date_minor_breaks = "1 week", date_labels = "%b") +
  #geom_vline(aes(xintercept = as.numeric(as.Date("2018-05-18"))),linetype=4)
  geom_rect(data = dat2,
            aes(xmin = x1, xmax = x2, ymin = y1, ymax = y2),
            fill = "blue", alpha = 0.2, 
            inherit.aes = FALSE) +
  annotate("text",x=as.Date("2017-12-28"), y=360, label= "(b)") 
one.wint
  
 


grid.arrange(norm.wint, one.wint, nrow = 1, ncol = 2)

####################
qdat.29<-subset(qdat, Date > "1928-9-30")
#total daily discharge   
  qdat.29$tdQcmd<-qdat.29$Qcms*86400 #cubic m / day

#day of water year, https://stackoverflow.com/questions/55525965/is-there-a-r-function-to-calculate-day-of-water-year
  hydro.day.new = function(x, start.month = 10L){
    start.yr = year(x) - (month(x) < start.month)
    start.date = make_date(start.yr, start.month, 1L)
    as.integer(x - start.date + 1L)
  }
  
 qdat.29$DOWY<-hydro.day.new(qdat.29$Date)
  tail(qdat.29)
#Cumulative discharge - cubic m/water year 
  qdat.29$CumWYQ<-99999
  #View(qdat.29)
  i=1
  for(i in 1:nrow(qdat.29)){
    if(qdat.29$DOWY[i]==1){
      qdat.29$CumWYQ[i]<-qdat.29$tdQcmd[i]
    }else{
      qdat.29$CumWYQ[i]<-qdat.29$CumWYQ[i-1] + qdat.29$tdQcmd[i]
    }
  }
 
d182.win<-subset(qdat.29, qdat.29$DOWY==182) #Mar 31
d31.win<-subset(qdat.29, qdat.29$DOWY==31)#Nov 1

summary(lm(CumWYQ~Year,data=d182.win))#R2=0.1595
win.wint<-lm(CumWYQ~Year,data=d182.win)
predict(win.wint)  
# max=915047128
# min=565185923
8980606618.27444/5786706125.49537 #Winooski winter cumulative discharge increased by 60% ~1.62
d182.win$Year #2022 is last year

pos_x <- 0.95
pos_y <- 0.1

connecticut_cumulative <- ggplot(d182.win,aes(Year,CumWYQ, color=Year)) + geom_point()  + 
  theme_bw()   +
  labs(y="Cumulative winter discharge (cubic meters)", x="Year") +
  geom_smooth(method="lm") + 
  scale_color_viridis_b() +
  annotation_custom(
    grid::textGrob("Connecticut River at Thompsonville", 
                   x = unit(0.02, "npc"), 
                   y = unit(0.92, "npc"),
                   just = c("left", "top"),
                   gp = gpar(col = "black", fontsize = 14))
  ) +
   annotation_custom(
    grid::textGrob(expression("R"^2==0.172), 
                   x = unit(0.02, "npc"), 
                   y = unit(0.88, "npc"),
                   just = c("left", "top"),
                   gp = gpar(col = "black", fontsize = 14))
  ) +
  annotation_custom(
    grid::textGrob("a)", 
                   x = unit(0.02, "npc"), 
                   y = unit(0.98, "npc"),
                   just = c("left", "top"),
                   gp = gpar(col = "black", fontsize = 16))
  ) +
   theme(text = element_text(size = 16))
                  
# mo2.cQ.182

# Save combined plot
ggsave(plot = connecticut_cumulative, width = 6, 
       height = 6, dpi = 300, 
       file = file.path(output_dir, "Connecticut_at_Thompsonville_1928-2025.jpg"))

connecticut_cumulative

Warning message:
“package ‘lubridate’ was built under R version 4.3.3”

Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


Warning message:
“package ‘ggplot2’ was built under R version 4.3.3”
Warning message:
“package ‘plyr’ was built under R version 4.3.3”
Warning message:
“package ‘tibble’ was built under R version 4.3.3”
Warning message:
“package ‘readr’ was built under R version 4.3.3”
Warning message:
“package ‘gridExtra’ was built under R version 4.3.3”
Warning message:
“package ‘tidyverse’ was built under R version 4.3.3”
Warning message:
“package ‘tidyr’ was built under R version 4.3.3”
Warning message:
“package ‘purrr’ was built under R version 4.3.3”
Warning message:
“package ‘dplyr’ was built under R version 4.3.3”
Warning message:
“package ‘stringr’ was built under R version 4.3.3”
Warning message:
“package ‘forcats’ was built under R version 4.3.3”
── Attaching core tidyverse packages ──────────────

ERROR: Error: object 'qdatX_00060_00003' not found
